# Pipeline Comparison — Raw vs Denoised Regression

**Central claim:** denoising noisy XRF spectra before regression improves
elemental concentration accuracy (lower MAE on active elements).

Two pipelines are evaluated on the same noisy test set:

| Pipeline | Description |
|----------|-------------|
| **Raw** | Noisy spectrum → Regressor (trained on noisy data) |
| **Denoised** | Noisy spectrum → Denoising model → Regressor (trained on clean data) |

Test spectra come from `GeneratorConfig.Presets.fast_scan()` (handheld XRF device scenario).

See `README.md` in `src/models/regression/` for the full experimental design.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

# Ensure project root is in sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.regression.generator import RegressionDataGenerator
from src.data.denoising.generator import DenoisingDataGenerator
from src.data.common.base_generator import GeneratorConfig
from src.models.regression import CNNRegressor, RegressionTrainer, evaluate_all
from src.models.denoising.architectures import CNNAutoencoder
from src.models.denoising.trainer import DenoisingTrainer

## Step 1 — Train Regressor on Clean Data

The *denoised pipeline* regressor is trained on high-quality, near-noiseless spectra
(`high_quality` preset). This is the model that will receive denoised inputs at inference.

The *raw pipeline* regressor is trained on noisy spectra (`fast_scan` preset) so it
has seen the noise distribution it will encounter at test time.

In [ ]:
SEED = 42
np.random.seed(SEED)

reg_gen = RegressionDataGenerator(seed=SEED)

# --- Regressor for the denoised pipeline (trained on clean spectra) ---
clean_config = GeneratorConfig.Presets.high_quality()
print("Generating clean training data for denoised-pipeline regressor...")
X_clean_train, y_clean_train = reg_gen.generate_dataset(2000, min_elements=2, max_elements=5, config=clean_config)
X_clean_val,   y_clean_val   = reg_gen.generate_dataset(400,  min_elements=2, max_elements=5, config=clean_config)

element_names     = y_clean_train.columns.tolist()
y_clean_train_np  = y_clean_train.values
y_clean_val_np    = y_clean_val.values

cnn_clean = CNNRegressor()
trainer_clean = RegressionTrainer(cnn_clean, learning_rate=1e-3)
print("Training clean regressor...")
history_clean = trainer_clean.train(
    X_clean_train, y_clean_train_np,
    X_clean_val,   y_clean_val_np,
    epochs=60, batch_size=64, patience=10,
)

# --- Regressor for the raw pipeline (trained on noisy spectra) ---
noisy_config = GeneratorConfig.Presets.fast_scan()
print("\nGenerating noisy training data for raw-pipeline regressor...")
X_noisy_train, y_noisy_train = reg_gen.generate_dataset(2000, min_elements=2, max_elements=5, config=noisy_config)
X_noisy_val,   y_noisy_val   = reg_gen.generate_dataset(400,  min_elements=2, max_elements=5, config=noisy_config)

y_noisy_train_np = y_noisy_train.values
y_noisy_val_np   = y_noisy_val.values

cnn_noisy = CNNRegressor()
trainer_noisy = RegressionTrainer(cnn_noisy, learning_rate=1e-3)
print("Training raw regressor...")
history_noisy = trainer_noisy.train(
    X_noisy_train, y_noisy_train_np,
    X_noisy_val,   y_noisy_val_np,
    epochs=60, batch_size=64, patience=10,
)

## Step 2 — Train Denoising Model

A 1D CNN autoencoder is trained on paired (noisy, clean) spectra.
The noisy input uses `fast_scan` settings; the clean target uses `high_quality` settings
(same element composition, different noise level — as generated by `DenoisingDataGenerator`).

In [ ]:
den_gen = DenoisingDataGenerator(seed=SEED)
den_config = GeneratorConfig.Presets.fast_scan()

print("Generating denoising training pairs...")
X_den_train_noisy, X_den_train_clean = den_gen.generate_dataset(2000, config=den_config)
X_den_val_noisy,   X_den_val_clean   = den_gen.generate_dataset(400,  config=den_config)

denoiser = CNNAutoencoder(input_dim=600)
den_trainer = DenoisingTrainer(denoiser, learning_rate=1e-3)

print("Training denoising model...")
den_history = den_trainer.train(
    X_den_train_noisy, X_den_train_clean,
    X_den_val_noisy,   X_den_val_clean,
    epochs=60, batch_size=64, patience=10,
)

plt.figure(figsize=(10, 4))
plt.plot(den_history["train_loss"], label="Train Loss")
plt.plot(den_history["val_loss"],   label="Val Loss")
plt.title("Denoising Model — Training Curve")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Step 3 — Generate Noisy Test Set

Test spectra are drawn from `fast_scan` (high noise, possible calibration drift).
Both pipelines are evaluated on this same test set.

In [ ]:
print("Generating noisy test set...")
X_test_noisy, y_test = reg_gen.generate_dataset(400, min_elements=2, max_elements=5, config=noisy_config)
y_test_np = y_test.values

print(f"Test set: X={X_test_noisy.shape}, y={y_test_np.shape}")

# Denoised version of the test set
X_test_denoised = den_trainer.predict(X_test_noisy)
print(f"Denoised test set: {X_test_denoised.shape}")

## Pipeline Evaluation

- **Raw pipeline**: noisy spectra → raw-trained regressor
- **Denoised pipeline**: noisy spectra → CNN denoiser → clean-trained regressor

In [ ]:
metrics_raw      = trainer_noisy.evaluate(X_test_noisy,    y_test_np, element_names=element_names)
metrics_denoised = trainer_clean.evaluate(X_test_denoised, y_test_np, element_names=element_names)

comparison = pd.DataFrame({
    "Raw": {k: v for k, v in metrics_raw.items()      if k != "per_element_mae"},
    "Denoised": {k: v for k, v in metrics_denoised.items() if k != "per_element_mae"},
}).T

comparison["MAE improvement"] = (
    (comparison["masked_mae"]["Raw"] - comparison["masked_mae"]["Denoised"])
    / comparison["masked_mae"]["Raw"] * 100
)

print("=== Pipeline Comparison on Noisy Test Set ===")
print(comparison.to_string(float_format="{:.4f}".format))

## Visual Comparison

For a few test samples: raw spectrum vs denoised spectrum, and the resulting
concentration predictions from both pipelines.

In [ ]:
preds_raw      = trainer_noisy.predict(X_test_noisy)
preds_denoised = trainer_clean.predict(X_test_denoised)
energies = np.arange(0, 30, 0.05)

fig, axes = plt.subplots(3, 2, figsize=(16, 13))

for i in range(3):
    ax_spec = axes[i, 0]
    ax_bar  = axes[i, 1]

    # Left: noisy vs denoised spectrum
    ax_spec.plot(energies, X_test_noisy[i],    label="Noisy",    alpha=0.6, color="red",       lw=1)
    ax_spec.plot(energies, X_test_denoised[i], label="Denoised", alpha=0.9, color="steelblue", lw=1.5)
    ax_spec.set_title(f"Sample {i} — Spectrum")
    ax_spec.set_xlabel("Energy (keV)")
    ax_spec.set_ylabel("Intensity")
    ax_spec.legend()
    ax_spec.grid(alpha=0.3)

    # Right: true vs raw-pred vs denoised-pred (active elements)
    active_mask = y_test_np[i] > 0
    active_els  = [element_names[j] for j in range(41) if active_mask[j]]
    true_vals   = y_test_np[i][active_mask]
    raw_vals    = preds_raw[i][active_mask]
    den_vals    = preds_denoised[i][active_mask]

    x_pos = np.arange(len(active_els))
    w = 0.25
    ax_bar.bar(x_pos - w,   true_vals, w, label="True",     color="steelblue", alpha=0.85)
    ax_bar.bar(x_pos,       raw_vals,  w, label="Raw",      color="tomato",    alpha=0.85)
    ax_bar.bar(x_pos + w,   den_vals,  w, label="Denoised", color="seagreen",  alpha=0.85)
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(active_els)
    ax_bar.set_ylabel("Relative Concentration")
    ax_bar.set_title(f"Sample {i} — Concentrations")
    ax_bar.legend()
    ax_bar.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Noise Level Ablation

How much does denoising help across different noise levels?

We sweep `n_counts_range` from very low noise to very high noise,
measure masked MAE for both pipelines, and plot the gap.

**Expected:** the denoised pipeline degrades more gracefully as noise increases.

In [ ]:
noise_levels = [500, 1000, 3000, 5000, 10000, 20000, 30000]
mae_raw_list      = []
mae_denoised_list = []

for n in noise_levels:
    cfg = GeneratorConfig.Presets.fast_scan()
    cfg.n_counts_range = (n, n)

    X_test_n, y_test_n = reg_gen.generate_dataset(200, min_elements=2, max_elements=5, config=cfg)
    y_test_n_np = y_test_n.values

    X_test_n_den = den_trainer.predict(X_test_n)

    m_raw = trainer_noisy.evaluate(X_test_n,     y_test_n_np)
    m_den = trainer_clean.evaluate(X_test_n_den, y_test_n_np)

    mae_raw_list.append(m_raw["masked_mae"])
    mae_denoised_list.append(m_den["masked_mae"])
    print(f"n_counts={n:6d}  |  Raw MAE={m_raw['masked_mae']:.4f}  |  Denoised MAE={m_den['masked_mae']:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(noise_levels, mae_raw_list,      marker="o", label="Raw pipeline",      color="tomato")
plt.plot(noise_levels, mae_denoised_list, marker="s", label="Denoised pipeline", color="seagreen")
plt.fill_between(noise_levels, mae_raw_list, mae_denoised_list, alpha=0.15, color="seagreen", label="Improvement")
plt.xscale("log")
plt.xlabel("Noise counts (n_counts)")
plt.ylabel("Masked MAE (active elements)")
plt.title("Regression MAE vs Noise Level — Raw vs Denoised Pipeline")
plt.legend()
plt.grid(alpha=0.3)
plt.show()